# Uncertainty Quantification Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Uncertainty Quantification (UQ)**.
It demonstrates how uncertainty in inputs, parameters, and models propagates to predictions.

Topics covered:

1. Aleatoric vs epistemic uncertainty
2. Linear approximation (first-order Taylor)
3. Monte Carlo uncertainty propagation — single input
4. Monte Carlo uncertainty propagation — multiple uncertain inputs
5. Confidence and prediction intervals
6. Sensitivity under uncertainty (correlation-based)
7. Variance decomposition and sensitivity indices
8. Ensemble model uncertainty
9. Parameter uncertainty in regression
10. Decision-making under uncertainty
11. Summary table
12. Mini exercises

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
np.random.seed(42)

## 1. Aleatoric vs Epistemic Uncertainty

Two fundamentally different types of uncertainty exist in any system:

| Type | Source | Reducible? | Example |
|------|---------|------------|--------|
| **Aleatoric** | Natural randomness | No | Sensor noise, material variability |
| **Epistemic** | Lack of knowledge | Yes | Limited data, imperfect model |

**Key insight:**
- Aleatoric uncertainty persists even with infinite data.
- Epistemic uncertainty decreases as we collect more data or improve the model.

In practice, both types appear simultaneously in predictions.

In [ ]:
# Simulate aleatoric: fixed noise floor
n_samples = [5, 20, 100, 500]
true_mean = 50.0
aleatoric_std = 5.0   # irreducible noise

rows = []
for n in n_samples:
    data = np.random.normal(true_mean, aleatoric_std, n)
    epistemic_std = data.std(ddof=1) / np.sqrt(n)   # SE of mean — reduces with n
    rows.append((n, round(aleatoric_std, 2), round(epistemic_std, 4)))

pd.DataFrame(rows, columns=['n', 'Aleatoric std (fixed)', 'Epistemic std (SE, reduces)'])

**Observation:** Aleatoric std stays constant; epistemic std (SE) shrinks as n grows.

## 2. Linear Approximation (First-Order Taylor)

For a function Y = f(X) with uncertain input X ~ (\mu_X, \sigma_X^2), a first-order
Taylor approximation propagates uncertainty analytically:

$$E[Y] \approx f(\mu_X)$$

$$\text{Var}(Y) \approx \left(\frac{df}{dX}\bigg|_{\mu_X}\right)^2 \sigma_X^2$$

For multiple inputs with covariance matrix \Sigma_X:

$$\text{Var}(Y) \approx J\, \Sigma_X\, J^T$$

where J is the Jacobian (gradient row vector) evaluated at the input means.

In [ ]:
# Example: Y = 3X + 0.5X^2,  X ~ N(mu_X, sigma_X^2)
mu_X    = 10.0
sigma_X = 2.0

# Analytical: dY/dX = 3 + X at X = mu_X
dY_dX = 3 + mu_X
E_Y_approx   = 3 * mu_X + 0.5 * mu_X**2
Var_Y_approx = dY_dX**2 * sigma_X**2

# Monte Carlo for comparison
X_mc = np.random.normal(mu_X, sigma_X, 50000)
Y_mc = 3 * X_mc + 0.5 * X_mc**2

pd.DataFrame({
    'Method':  ['Linear approx (Taylor)', 'Monte Carlo (reference)'],
    'E[Y]':    [round(E_Y_approx, 4),   round(Y_mc.mean(), 4)],
    'Var[Y]':  [round(Var_Y_approx, 4), round(Y_mc.var(ddof=1), 4)],
    'Std[Y]':  [round(np.sqrt(Var_Y_approx), 4), round(Y_mc.std(ddof=1), 4)]
})

**Practice:** The Taylor approximation is exact for linear functions and a first-order approximation
for nonlinear ones. Try increasing sigma_X to see where the approximation breaks down.

## 3. Monte Carlo Uncertainty Propagation — Single Input

The most practical UQ method:
1. Sample X^{(i)} from the input distribution
2. Evaluate Y^{(i)} = f(X^{(i)})
3. Summarize the output distribution

No assumptions about the function's shape are needed.

In [ ]:
# Nonlinear system: Y = 3X + 0.5X^2,  X ~ N(10, 2^2)
N = 5000
X_single = np.random.normal(mu_X, sigma_X, N)
Y_single = 3 * X_single + 0.5 * X_single**2

plt.figure(figsize=(8, 4))
plt.hist(Y_single, bins=40)
plt.title('Monte Carlo Output Distribution (single input)')
plt.xlabel('Y = 3X + 0.5X^2')
plt.ylabel('Frequency')
plt.show()

pd.DataFrame({
    'Statistic': ['Mean Y', 'Std Y', '5th percentile', '95th percentile'],
    'Value':     [Y_single.mean(), Y_single.std(ddof=1),
                  np.percentile(Y_single, 5), np.percentile(Y_single, 95)]
})

## 4. Monte Carlo Uncertainty Propagation — Multiple Uncertain Inputs

Most real systems have multiple uncertain inputs.

For Y = f(X_1, X_2, ..., X_p) with uncertain X_i ~ P_i:
1. Sample all inputs jointly
2. Evaluate f for each sample
3. Study the output distribution

If inputs are correlated, joint sampling must respect the covariance structure.

In [ ]:
# Y = a*X1 + b*X2 + c*X3,  independent uncertain inputs
a, b, c = 2.0, -1.5, 0.8
mu_inputs  = np.array([5.0, 3.0, 10.0])
std_inputs = np.array([1.0, 0.5, 2.0])    # different uncertainty levels

N_mc = 8000
X1 = np.random.normal(mu_inputs[0], std_inputs[0], N_mc)
X2 = np.random.normal(mu_inputs[1], std_inputs[1], N_mc)
X3 = np.random.normal(mu_inputs[2], std_inputs[2], N_mc)
Y_multi = a*X1 + b*X2 + c*X3

# Analytical reference for linear system
E_Y_analytical   = a*mu_inputs[0] + b*mu_inputs[1] + c*mu_inputs[2]
Var_Y_analytical = a**2*std_inputs[0]**2 + b**2*std_inputs[1]**2 + c**2*std_inputs[2]**2

pd.DataFrame({
    'Method':  ['MC', 'Analytical'],
    'E[Y]':    [round(Y_multi.mean(), 4), round(E_Y_analytical, 4)],
    'Var[Y]':  [round(Y_multi.var(ddof=1), 4), round(Var_Y_analytical, 4)]
})

## 5. Confidence and Prediction Intervals

**Confidence interval** — for the estimated mean:

$$\bar{y} \pm 1.96\,\frac{s}{\sqrt{n}}$$

**Prediction interval** — for a new single observation:

$$\hat{y} \pm 1.96\,s\sqrt{1 + \frac{1}{n}}$$

Prediction intervals are always wider than confidence intervals because they include individual variability.

In [ ]:
sample = np.random.normal(50, 5, 100)
mean_s = sample.mean()
std_s  = sample.std(ddof=1)
n_s    = len(sample)

ci_low  = mean_s - 1.96 * std_s / np.sqrt(n_s)
ci_high = mean_s + 1.96 * std_s / np.sqrt(n_s)
pi_low  = mean_s - 1.96 * std_s * np.sqrt(1 + 1/n_s)
pi_high = mean_s + 1.96 * std_s * np.sqrt(1 + 1/n_s)

pd.DataFrame({
    'Interval':  ['95% Confidence (mean)', '95% Prediction (new obs)'],
    'Lower':     [ci_low,  pi_low],
    'Upper':     [ci_high, pi_high],
    'Width':     [ci_high - ci_low, pi_high - pi_low]
})

## 6. Sensitivity Under Uncertainty

Sensitivity analysis identifies which uncertain inputs matter most for the output.

A simple rank: the Pearson correlation between each input and the output.

In [ ]:
X1_s = np.random.normal(5, 1,   3000)
X2_s = np.random.normal(2, 0.2, 3000)
X3_s = np.random.normal(8, 3,   3000)   # high std but small coefficient
Y_s  = 4*X1_s + 0.5*X2_s + 0.1*X3_s

corr1 = np.corrcoef(X1_s, Y_s)[0, 1]
corr2 = np.corrcoef(X2_s, Y_s)[0, 1]
corr3 = np.corrcoef(X3_s, Y_s)[0, 1]

sens_df = pd.DataFrame({
    'Input':      ['X1 (a=4, std=1)', 'X2 (a=0.5, std=0.2)', 'X3 (a=0.1, std=3)'],
    'Correlation': [corr1, corr2, corr3]
}).sort_values('Correlation', key=abs, ascending=False)
sens_df

## 7. Variance Decomposition and Sensitivity Indices

For a linear model with independent inputs, the fraction of output variance explained by input X_i is:

$$S_i = \frac{a_i^2 \sigma_i^2}{\text{Var}(Y)}$$

This is a simplified first-order Sobol sensitivity index for linear models.
It shows how much of the total output uncertainty each input is responsible for.

In [ ]:
coeffs = np.array([4.0, 0.5, 0.1])
stds   = np.array([1.0, 0.2, 3.0])
input_var_contrib = coeffs**2 * stds**2
total_var = input_var_contrib.sum()
sobol_s   = input_var_contrib / total_var

sobol_df = pd.DataFrame({
    'Input':        ['X1', 'X2', 'X3'],
    'Coefficient':  coeffs,
    'Input std':    stds,
    'Var contrib':  input_var_contrib,
    'Sensitivity S_i': sobol_s
})
sobol_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(sobol_df['Input'], sobol_df['Sensitivity S_i'])
plt.title('Sensitivity Indices (fraction of output variance)')
plt.ylabel('S_i')
plt.show()

## 8. Ensemble Model Uncertainty

Ensemble methods provide epistemic uncertainty estimates by training multiple models and
measuring their disagreement.

Wide spread among ensemble members signals high epistemic uncertainty in that region.

In [ ]:
x_ens = np.linspace(0, 10, 100)
preds = []
for i in range(5):
    noise_level = 1 + i * 0.3
    preds.append(2*x_ens + 3 + np.random.normal(0, noise_level, len(x_ens)))
preds = np.array(preds)

ensemble_mean = preds.mean(axis=0)
ensemble_std  = preds.std(axis=0, ddof=1)

plt.figure(figsize=(8, 4))
plt.plot(x_ens, ensemble_mean, label='Ensemble mean')
plt.fill_between(x_ens, ensemble_mean - 2*ensemble_std,
                         ensemble_mean + 2*ensemble_std, alpha=0.3, label='Mean +/- 2 std')
for i, p in enumerate(preds):
    plt.plot(x_ens, p, alpha=0.15, linewidth=0.8)
plt.title('Ensemble Prediction with Uncertainty Band')
plt.xlabel('x')
plt.ylabel('Prediction')
plt.legend()
plt.show()

pd.DataFrame({'x': x_ens[:5], 'mean_pred': ensemble_mean[:5], 'std_uncertainty': ensemble_std[:5]})

## 9. Parameter Uncertainty in Regression

Regression coefficients are uncertain. Their covariance matrix is:

$$\text{Cov}(\hat{\beta}) = \sigma^2 (X^T X)^{-1}$$

Diagonal entries give the variance of each coefficient estimate.
This uncertainty propagates to uncertainty in predictions.

In [ ]:
Xr = np.linspace(0, 10, 80).reshape(-1, 1)
yr = 4 * Xr[:, 0] + 2 + np.random.normal(0, 2, 80)

model = LinearRegression().fit(Xr, yr)
pred  = model.predict(Xr)
resid = yr - pred
sigma2 = np.var(resid, ddof=2)

X_design = np.column_stack([np.ones(len(Xr)), Xr[:, 0]])
cov_beta = sigma2 * np.linalg.inv(X_design.T @ X_design)
se_intercept = np.sqrt(cov_beta[0, 0])
se_slope     = np.sqrt(cov_beta[1, 1])

pd.DataFrame({
    'Parameter':  ['Intercept', 'Slope'],
    'Estimate':   [model.intercept_, model.coef_[0]],
    'Std Error':  [se_intercept, se_slope],
    '95% CI low': [model.intercept_ - 1.96*se_intercept, model.coef_[0] - 1.96*se_slope],
    '95% CI high':[model.intercept_ + 1.96*se_intercept, model.coef_[0] + 1.96*se_slope]
})

## 10. Decision-Making Under Uncertainty

When choosing actions under uncertainty, we compute the expected utility:

$$EU(a) = \sum_{s} P(s) \cdot U(a, s)$$

The rational choice is the action with the highest expected utility.

In [ ]:
# Two actions, two scenarios (good / bad market)
prob_scenarios = np.array([0.7, 0.3])     # P(good), P(bad)
utility_A      = np.array([100, 20])       # Action A: high upside, moderate downside
utility_B      = np.array([70,  50])       # Action B: moderate upside, safer downside
utility_C      = np.array([60,  60])       # Action C: certain moderate outcome

EU_A = np.dot(prob_scenarios, utility_A)
EU_B = np.dot(prob_scenarios, utility_B)
EU_C = np.dot(prob_scenarios, utility_C)

result_df = pd.DataFrame({
    'Action':           ['A', 'B', 'C'],
    'Good scenario':    utility_A.tolist()  + [utility_C[0]],
    'Bad scenario':     utility_A.tolist()  + [utility_C[1]],
    'Expected Utility': [EU_A, EU_B, EU_C]
})

# Fix the table data
result_df = pd.DataFrame({
    'Action':           ['A', 'B', 'C'],
    'Utility (good)':   [utility_A[0], utility_B[0], utility_C[0]],
    'Utility (bad)':    [utility_A[1], utility_B[1], utility_C[1]],
    'Expected Utility': [EU_A, EU_B, EU_C]
})
result_df

## 11. Summary Table

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'Taylor E[Y] approximation',
        'Taylor Std[Y] approximation',
        'MC E[Y] (single input)',
        'MC Std[Y] (single input)',
        'CI width (95%)',
        'PI width (95%)',
        'Most sensitive input',
        'Dominant sensitivity index S_i',
        'Average ensemble std',
        'Best decision'
    ],
    'Value': [
        round(E_Y_approx, 3),
        round(np.sqrt(Var_Y_approx), 3),
        round(Y_single.mean(), 3),
        round(Y_single.std(ddof=1), 3),
        round(ci_high - ci_low, 4),
        round(pi_high - pi_low, 4),
        'X1',
        round(sobol_s.max(), 4),
        round(ensemble_std.mean(), 4),
        'A' if EU_A >= max(EU_B, EU_C) else ('B' if EU_B >= EU_C else 'C')
    ]
})
summary

## 12. Mini Exercises

Try these on your own:

1. Increase the input standard deviation in the Taylor approximation and measure the relative error versus Monte Carlo.
2. Add a correlated input pair (X1 and X2 with correlation 0.8) in the multi-input MC and compare variance to the independent case.
3. Compute sensitivity indices for a model with five inputs and rank them by contribution to output variance.
4. Vary the sample size n in the confidence-interval cell and plot how the CI width changes.
5. Add more ensemble members to the ensemble uncertainty cell and observe how the band stabilizes.
6. Replace the decision table with a three-scenario model and recompute expected utilities.
7. Fit a polynomial regression model and compute parameter uncertainty for all coefficients.
8. Apply Monte Carlo propagation to a real engineering formula (e.g., beam deflection with uncertain load and moment of inertia).

These exercises are especially useful for AI, machine learning, digital twins, structural engineering, and risk analysis.